# peak-plus-edge — SpectraFit Example Notebook

This user-facing notebook is the runnable companion for the `examples/peak-plus-edge` workflow.

Pseudo-Voigt peak riding on a smooth erf edge and constant offset.

## Workflow
1. Resolve local notebook paths.
2. Load `data.csv` through `sf.read(...)`.
3. Edit compact `sf.peak(...)` and `sf.background(...)` definitions.
4. Run `sf.fit(...)` and inspect the inline plot/metrics.
5. Export bundled live notebook artifacts under `outputs/live/notebook/`.


In [1]:
from __future__ import annotations

from pathlib import Path

import spectrafit.notebook as sf


## 1 — Resolve local paths

This notebook always loads `data.csv` from the local notebook directory and writes exports under `outputs/live/notebook/`.

In [2]:
NOTEBOOK_ROOT = Path.cwd()
DATA_PATH = NOTEBOOK_ROOT / 'data.csv'
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs" / "live" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook root: {NOTEBOOK_ROOT}")
print(f"Using local data file: {DATA_PATH}")


Notebook root: /Users/hahn/LocalDocuments/GitHub_Forks/spectrafit/examples/peak-plus-edge
Using local data file: /Users/hahn/LocalDocuments/GitHub_Forks/spectrafit/examples/peak-plus-edge/data.csv


## 2 — Load the local dataset

Use the single notebook import to load the local spectrum and keep the data columns attached to the dataframe for the fit step.

In [3]:
df = sf.read(DATA_PATH, x='energy', y='intensity')
location = NOTEBOOK_ROOT.name or str(NOTEBOOK_ROOT)
print(f"Loaded {DATA_PATH.name} with {len(df)} rows from {location}")
df.head()


Loaded data.csv with 280 rows from peak-plus-edge


,energy,intensity
0,-3.500000,0.065238
1,-3.474910,0.049121
2,-3.449821,0.070626
3,-3.424731,0.072927
4,-3.399642,0.038248


## 3 — Define the fit and run it

The compact notebook API still compiles into the canonical `UnifiedFittingConfig -> FittingPipeline -> FitResult` chain under the hood, but the cell below hides the internal object graph.

In [4]:
peaks = [
    sf.peak(
        'pseudovoigt',
        id='peak1',
        amplitude=(0.9, 0.0, 2.0),
        center=(0.65, 0.0, 1.5),
        fwhmg=(0.35, 0.1, 1.0),
        fwhml=(0.3, 0.1, 1.0),
    ),
    sf.peak(
        'erf',
        id='edge',
        amplitude=(0.4, 0.0, 1.0),
        center=(-0.3, -1.0, 0.5),
        sigma=(0.45, 0.1, 1.2),
    ),
]

background = [
    sf.background(
        'constant',
        id='bg',
        amplitude=(0.06, 0.0, 0.2),
    ),
]

optimizer = sf.OptimizerConfig(
    max_nfev=1200,
    method='leastsq',
)

result = sf.fit(
    df,
    peaks=peaks,
    background=background,
    optimizer=optimizer,
    name='peak-plus-edge',
)

result.plot()
result.metrics

,chi_square,reduced_chi_square,akaike_information,bayesian_information,explained_variance_score,r2_score,max_error,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,mean_absolute_percentage_error,mean_poisson_deviance
0,0.0346,0.000127,-2503.637373,-2474.559056,0.999428,0.999428,0.035108,0.008844,0.000124,0.00007,0.007295,0.061878,0.000729


## 4 — Export notebook artifacts

Use one `result.save(...)` call to write the fitted dataframe, metric table, peak table, HTML plot, report, and lockfile.

In [5]:
artifacts = result.save(OUTPUT_DIR, name='peak-plus-edge')

[path.name for path in artifacts]


['fit_peak-plus-edge.csv',
 'metric_peak-plus-edge.csv',
 'peaks_peak-plus-edge.csv',
 'fit_peak-plus-edge.html',
 'report_peak-plus-edge.toml',
 'peak-plus-edge.lock']

## Next steps

- Inspect the generated CSVs, HTML fit plot, report, and lockfile in `outputs/live/notebook/`.
- Edit the compact `sf.peak(...)` / `sf.background(...)` definitions and rerun the fit cell to explore different models, bounds, and solver settings.
